# 39. Combination Sum
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/combination-sum/

## 💡 Concepts

**Core concept(s):** **Backtracking** — build combinations choice by choice, undoing when you overshoot.

**Why it applies here:** We need *all* combinations that sum to the target, and each number may repeat. Backtracking explores a tree of choices: add a candidate, recurse on the smaller remainder (allowed to reuse it), then remove it and try the next. Sorting lets us stop early when a candidate already exceeds the remainder.

**Key intuition:** Pick a candidate, subtract it, keep going with the same or later candidates; back off when you overshoot or hit zero.

---

### 📚 What is Backtracking?
**Backtracking** builds a solution step by step, and **undoes** a step when it can't lead anywhere. It explores a tree of choices, pruning branches that overshoot.
- **Complexity:** exponential in the worst case; pruning (sorting, early breaks) keeps it practical.

### 📚 Subproblems & Recurrence
The heart of DP is a **recurrence**: the answer for a state written in terms of smaller states (e.g. `dp[i] = dp[i-1] + dp[i-2]`). Find the recurrence and the base cases, and the code writes itself.

---

**Prerequisite knowledge:**
- Recursion that appends then pops (undo).
- A start index to avoid duplicate combinations.

## 📝 Problem

Given distinct positive `candidates` and a `target`, return all unique combinations that sum to it. Each candidate may be used unlimited times.

**Example**
```
candidates=[2,3,6,7], target=7 -> [[2,2,3],[7]]
```

> One approach: backtracking (with sorting + early pruning).

### Approach — Backtracking

**Idea:** Sort candidates. Recurse carrying the remaining target and a start index (so we never go backwards, avoiding duplicate sets). Adding the same index again allows reuse. Break when a candidate exceeds the remainder.

**Time:** exponential in the number of combinations. **Space:** `O(target / min candidate)` recursion depth.

In [ ]:
def combination_sum(candidates, target):
    candidates = sorted(candidates)        # sorting lets us stop early when a number is too big
    res = []
    def backtrack(start, remain, path):    # remain = target left to reach; path = chosen so far
        if remain == 0:
            res.append(path[:])            # exact hit -> save a COPY of the combination
            return
        for i in range(start, len(candidates)):   # only consider candidates from `start` onward
            if candidates[i] > remain:
                break                      # sorted -> everything after is also too big
            path.append(candidates[i])     # choose this candidate
            backtrack(i, remain - candidates[i], path)   # i (not i+1) allows reusing it
            path.pop()                     # un-choose it (backtrack) and try the next
    backtrack(0, target, [])
    return res

In [ ]:
# Correctness check
def norm(combos):
    return sorted(tuple(sorted(c)) for c in combos)
tests = [
    ([2,3,6,7],7,[[2,2,3],[7]]),
    ([2,3,5],8,[[2,2,2,2],[2,3,3],[3,5]]),
    ([2],1,[]),
]
for cand, tgt, exp in tests:
    got = combination_sum(cand, tgt)
    print(f"{cand}, target={tgt} -> {got}")
    assert norm(got) == norm(exp), "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

*(Backtracking is inherently exponential; we grow the target on a small candidate set.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return ([2, 3, 5, 7], n)   # more combinations as the target grows
solutions = {
    "backtracking (exponential)": combination_sum,
}
sizes = [12, 16, 20, 24]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Backtracking = choose / explore / undo:** the template for "list all combinations/permutations/subsets".
- **Start index kills duplicates:** never revisit earlier candidates → each set counted once.
- **Prune with sorting:** break as soon as a candidate exceeds the remainder.
- **Signal:** "find all combinations that ...", "subsets summing to X", "reuse allowed".
- **Related problems:** Combination Sum II/III/IV, Subsets, Permutations, Palindrome Partitioning.
- **Common pitfalls:** (1) `i+1` vs `i` (reuse or not); (2) appending the shared list without copying.